In [3]:
import os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms
from datasets import load_dataset, load_from_disk
import wandb
import numpy as np
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt

# -----------------------------
# wandb init
# -----------------------------
wandb.init(
    project="brain-tumor-autoencoder-anomaly",
    config={
        "latent_dim": 128,
        "batch_size": 64,
        "lr_ae": 1e-3,
        "epochs_ae": 20,   #25
        "img_size": 128,
        "threshold_percentile": 95,

        # UNet hyperparameters
        "base_channels": 32,
        "num_blocks": 4,
        "kernel_size": 3,
        "activation": "leakyrelu",
        "norm": "batch",
        "dropout": 0.1,
        "output_activation": "tanh"
    }
)
config = wandb.config

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -----------------------------
# Dataset (download + local cache)
# -----------------------------
DATASET_NAME = "PranomVignesh/MRI-Images-of-Brain-Tumor"
LOCAL_DATASET_DIR = "hf_brain_tumor_dataset"

def load_or_download_dataset():
    if os.path.exists(LOCAL_DATASET_DIR):
        print("📂 Loading dataset from local disk...")
        return load_from_disk(LOCAL_DATASET_DIR)
    else:
        print("⬇️ Downloading dataset from Hugging Face...")
        ds = load_dataset(DATASET_NAME)
        ds.save_to_disk(LOCAL_DATASET_DIR)
        print(f"💾 Dataset saved locally to: {LOCAL_DATASET_DIR}")
        return ds

dataset = load_or_download_dataset()

transform = transforms.Compose([
    transforms.Resize((config.img_size, config.img_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])



AE_train_loss,█▂▂▁▁▁▁▁▁▁
AE_val_loss,█▃▂▁▁▁▁▁▁▁
epoch,▁▂▃▃▄▅▆▆▇█
threshold,▁
AE_train_loss,0.00585
AE_val_loss,0.0014
epoch,9
threshold,0.00217


📂 Loading dataset from local disk...


In [5]:
class HFBrainDataset(torch.utils.data.Dataset):
    def __init__(self, hf_split, transform=None):
        self.data = hf_split
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img = self.data[idx]["image"].convert("L")
        label = self.data[idx]["label"]
        if self.transform:
            img = self.transform(img)
        return img, label

# Correct label: 3 = no tumor
def filter_no_tumor(hf_split):
    return [item for item in hf_split if item["label"] == 3]

train_no_tumor = filter_no_tumor(dataset["train"])
val_no_tumor   = filter_no_tumor(dataset["validation"])

train_ds = HFBrainDataset(train_no_tumor, transform)
val_ds   = HFBrainDataset(val_no_tumor, transform)
test_ds  = HFBrainDataset(dataset["test"], transform)

train_loader = DataLoader(train_ds, batch_size=config.batch_size, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=config.batch_size, shuffle=False)
test_loader  = DataLoader(test_ds, batch_size=config.batch_size, shuffle=False)

# -----------------------------
# UNet-style Autoencoder
# -----------------------------
class ConvBlock(nn.Module):
    def __init__(self, in_c, out_c, kernel_size=3, activation="relu", norm="batch", dropout=0.0):
        super().__init__()

        act = {
            "relu": nn.ReLU,
            "leakyrelu": lambda: nn.LeakyReLU(0.2),
            "elu": nn.ELU,
            "gelu": nn.GELU
        }[activation]

        def norm_layer(c):
            if norm == "batch":
                return nn.BatchNorm2d(c)
            elif norm == "layer":
                return nn.GroupNorm(1, c)
            else:
                return nn.Identity()

        layers = [
            nn.Conv2d(in_c, out_c, kernel_size, padding=kernel_size // 2),
            norm_layer(out_c),
            act(),
        ]
        if dropout > 0:
            layers.append(nn.Dropout2d(dropout))

        layers += [
            nn.Conv2d(out_c, out_c, kernel_size, padding=kernel_size // 2),
            norm_layer(out_c),
            act(),
        ]
        if dropout > 0:
            layers.append(nn.Dropout2d(dropout))

        self.block = nn.Sequential(*layers)

    def forward(self, x):
        return self.block(x)


class UNetAutoencoder(nn.Module):
    def __init__(
        self,
        img_channels=1,
        base_channels=32,
        num_blocks=4,
        latent_dim=128,
        kernel_size=3,
        activation="relu",
        norm="batch",
        dropout=0.0,
        output_activation="tanh"
    ):
        super().__init__()

        self.num_blocks = num_blocks

        # Encoder path
        enc_channels = []
        in_c = img_channels
        self.enc_blocks = nn.ModuleList()
        self.pools = nn.ModuleList()

        for i in range(num_blocks):
            out_c = base_channels * (2 ** i)
            self.enc_blocks.append(
                ConvBlock(in_c, out_c, kernel_size, activation, norm, dropout)
            )
            enc_channels.append(out_c)
            self.pools.append(nn.MaxPool2d(2))
            in_c = out_c

        # Bottleneck
        bottleneck_c = base_channels * (2 ** num_blocks)
        self.bottleneck = ConvBlock(in_c, bottleneck_c, kernel_size, activation, norm, dropout)

        # Latent projection (optional, keeps your latent_dim config)
        self._dummy = torch.zeros(1, img_channels, 128, 128)
        with torch.no_grad():
            x = self._dummy
            for i in range(num_blocks):
                x = self.enc_blocks[i](x)
                x = self.pools[i](x)
            x = self.bottleneck(x)
        self.bottleneck_shape = x.shape  # (1, C, H, W)
        flat_dim = x.numel()
        self.enc_fc = nn.Linear(flat_dim, latent_dim)
        self.dec_fc = nn.Linear(latent_dim, flat_dim)

        # Decoder path (UNet-style with skip connections)
        self.upconvs = nn.ModuleList()
        self.dec_blocks = nn.ModuleList()

        dec_in_c = bottleneck_c
        for i in reversed(range(num_blocks)):
            skip_c = enc_channels[i]
            out_c = skip_c
            self.upconvs.append(
                nn.ConvTranspose2d(dec_in_c, out_c, kernel_size=2, stride=2)
            )
            self.dec_blocks.append(
                ConvBlock(dec_in_c, out_c, kernel_size, activation, norm, dropout)
            )
            dec_in_c = out_c

        # Final conv
        self.final_conv = nn.Conv2d(base_channels, img_channels, kernel_size=1)

        if output_activation == "tanh":
            self.out_act = nn.Tanh()
        elif output_activation == "sigmoid":
            self.out_act = nn.Sigmoid()
        else:
            self.out_act = nn.Identity()

    def encode(self, x):
        skips = []
        for i in range(self.num_blocks):
            x = self.enc_blocks[i](x)
            skips.append(x)
            x = self.pools[i](x)
        x = self.bottleneck(x)
        x_flat = x.view(x.size(0), -1)
        z = self.enc_fc(x_flat)
        return z, skips

    def decode(self, z, skips):
        x = self.dec_fc(z)
        x = x.view(-1, *self.bottleneck_shape[1:])

        for i in range(self.num_blocks):
            x = self.upconvs[i](x)
            skip = skips[self.num_blocks - 1 - i]
            # handle possible size mismatch due to pooling/upsampling
            if x.shape[-2:] != skip.shape[-2:]:
                diff_y = skip.size(2) - x.size(2)
                diff_x = skip.size(3) - x.size(3)
                x = nn.functional.pad(
                    x,
                    [diff_x // 2, diff_x - diff_x // 2,
                     diff_y // 2, diff_y - diff_y // 2]
                )
            x = torch.cat([skip, x], dim=1)
            x = self.dec_blocks[i](x)

        x = self.final_conv(x)
        return self.out_act(x)

    def forward(self, x):
        z, skips = self.encode(x)
        x_hat = self.decode(z, skips)
        return x_hat, z

# Instantiate model
ae = UNetAutoencoder(
    img_channels=1,
    base_channels=config.base_channels,
    num_blocks=config.num_blocks,
    latent_dim=config.latent_dim,
    kernel_size=config.kernel_size,
    activation=config.activation,
    norm=config.norm,
    dropout=config.dropout,
    output_activation=config.output_activation
).to(device)

criterion_rec = nn.MSELoss()
optimizer_ae = torch.optim.Adam(ae.parameters(), lr=config.lr_ae)

# -----------------------------
# Reconstruction error helper
# -----------------------------
def reconstruction_errors(model, loader):
    model.eval()
    errors = []
    labels = []
    with torch.no_grad():
        for imgs, lbls in loader:
            imgs = imgs.to(device)
            recon, _ = model(imgs)
            err = torch.mean((recon - imgs) ** 2, dim=[1, 2, 3]).cpu().numpy()
            errors.extend(err)
            labels.extend(lbls.numpy())
    return np.array(errors), np.array(labels)

# -----------------------------
# Train Autoencoder
# -----------------------------
for epoch in range(config.epochs_ae):
    ae.train()
    total_loss = 0

    for imgs, _ in train_loader:
        imgs = imgs.to(device)
        optimizer_ae.zero_grad()
        recon, _ = ae(imgs)
        loss = criterion_rec(recon, imgs)
        loss.backward()
        optimizer_ae.step()
        total_loss += loss.item() * imgs.size(0)

    train_loss = total_loss / len(train_ds)
    val_errors, _ = reconstruction_errors(ae, val_loader)
    val_loss = val_errors.mean()

    wandb.log({
        "AE_train_loss": train_loss,
        "AE_val_loss": val_loss,
        "epoch": epoch
    })

    print(f"Epoch {epoch+1}/{config.epochs_ae} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

# -----------------------------
# Threshold selection
# -----------------------------
val_errors, _ = reconstruction_errors(ae, val_loader)
threshold = np.percentile(val_errors, config.threshold_percentile)
wandb.log({"threshold": threshold})
print(f"Chosen threshold: {threshold:.6f}")

# -----------------------------
# Evaluate on test set
# -----------------------------
test_errors, test_labels = reconstruction_errors(ae, test_loader)

preds = (test_errors > threshold).astype(int)
true = (test_labels != 3).astype(int)

cm = confusion_matrix(true, preds)
print("Confusion matrix:")
print(cm)

wandb.log({
    "anomaly_confusion_matrix": wandb.plot.confusion_matrix(
        y_true=true,
        preds=preds,
        class_names=["no_tumor", "tumor"]
    )
})

# -----------------------------
# Visualize detected tumors
# -----------------------------
def show_detected_tumors(model, loader, threshold, max_images=5):
    model.eval()
    shown = 0

    for imgs, labels in loader:
        imgs = imgs.to(device)
        recon, _ = model(imgs)

        pixel_err = (recon - imgs) ** 2
        img_err = pixel_err.mean(dim=[1,2,3]).detach().cpu().numpy()

        for i in range(len(imgs)):
            if img_err[i] > threshold:
                orig = imgs[i].detach().cpu()
                rec  = recon[i].detach().cpu()
                heat = pixel_err[i].detach().cpu()

                heat = (heat - heat.min()) / (heat.max() - heat.min() + 1e-8)

                fig, ax = plt.subplots(1, 3, figsize=(12,4))
                ax[0].imshow(orig.squeeze(), cmap="gray")
                ax[0].set_title("Original")

                ax[1].imshow(rec.squeeze(), cmap="gray")
                ax[1].set_title("Reconstruction")

                ax[2].imshow(orig.squeeze(), cmap="gray")
                ax[2].imshow(heat.squeeze(), cmap="jet", alpha=0.5)
                ax[2].set_title("Anomaly Heatmap")

                for a in ax:
                    a.axis("off")

                plt.show()

                wandb.log(
                    {
                        "Detected_Tumor": [
                            wandb.Image(orig, caption="Original"),
                            wandb.Image(rec, caption="Reconstruction"),
                            wandb.Image(heat, caption="Heatmap"),
                        ]
                    }
                )

                shown += 1
                if shown >= max_images:
                    return

# Show 10 detected tumors
show_detected_tumors(ae, test_loader, threshold, max_images=10)

# -----------------------------
# Save model
# -----------------------------
torch.save(ae.state_dict(), "autoencoder_unet.pt")
wandb.save("autoencoder_unet.pt")

print("Done.")


Output hidden; open in https://colab.research.google.com to view.